# 02 ? Running Experiments

A standard qubit characterisation sequence using a live QICK session:

| Step | Experiment | Extracts |
|---|---|---|
| 1 | `ResonatorSpec` | Resonator frequency, kappa |
| 2 | `QubitSpec` | Qubit ge transition frequency |
| 3 | `PowerRabi` | pi-pulse gain |
| 4 | `T1` | Energy relaxation time |
| 5 | `Ramsey` | T2*, detuning |


In [ ]:
import sys; sys.path.insert(0, '../')
import numpy as np
import matplotlib.pyplot as plt
from qick.asm_v2 import QickSweep1D

from QickworkspaceV2 import BaseExperiment, ExperimentConfig
from QickworkspaceV2.config.system_cfg import config_list

BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82',
    ns_port=8888,
    proxy_name='myqick',
    data_path=r'D:\Labber_Data\Jay\test',
)

qubit = 'Q1'
cfg_all = ExperimentConfig(config_list)


## Helper: plot a result

In [ ]:
def show(result, title='', xlabel='x', ylabel='Signal (a.u.)'):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(result.x_axis, np.abs(result.raw_iq), lw=1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    print('fit_result:', result.fit_result)
    print()

## Step 1 — Resonator Spectroscopy (`ResonatorSpec`)

Sweeps the readout tone frequency.  The circle fit (ABCD hanger model) extracts
`f0` (resonator frequency), `κ` (total linewidth), and `κ_c` (coupling rate).

In [ ]:
from QickworkspaceV2.experiments.resonator import ResonatorSpec

center = cfg_all.get_qubit(qubit)['res_freq_ge']
cfg = cfg_all.get_qubit(qubit)
cfg.update([
    ('steps', 101),
    ('res_freq_ge', QickSweep1D('freqloop', center - 10, center + 10)),
    ('relax_delay', 0),
])
res_result = ResonatorSpec(cfg).run(py_avg=5)
show(res_result, title='Resonator Spec', xlabel='Frequency (MHz)')

if res_result.scalar_result is not None:
    cfg_all.update('res_freq_ge', res_result.scalar_result, q_index=qubit)


## Step 2 — Qubit Spectroscopy (`QubitSpec`)

Sweeps the qubit drive frequency while the resonator is monitored.  A Lorentzian
fit locates the ge transition frequency `f_qubit`.

In [ ]:
from QickworkspaceV2.experiments.qubit_ge import QubitSpec

center = cfg_all.get_qubit(qubit)['qb_freq_ge']
cfg = cfg_all.get_qubit(qubit)
cfg.update([
    ('steps', 101),
    ('qb_freq_ge', QickSweep1D('freqloop', center - 50, center + 50)),
    ('qb_mixer', center),
    ('qb_gain_ge', 0.1),
    ('qb_flat_top_length_ge', 1.0),
    ('relax_delay', 1),
])
qs_result = QubitSpec(cfg).run(py_avg=5)
show(qs_result, title='Qubit Spectroscopy', xlabel='Drive frequency (MHz)')

if qs_result.scalar_result is not None:
    cfg_all.update('qb_freq_ge', qs_result.scalar_result, q_index=qubit)
    cfg_all.update('qb_mixer', qs_result.scalar_result, q_index=qubit)


## Step 3 — Power Rabi (`PowerRabi`)

Sweeps the qubit drive amplitude.  The cosine fit finds the gain that produces
a π rotation (`pi_gain_ge`).

In [ ]:
from QickworkspaceV2.experiments.qubit_ge import PowerRabi

cfg = cfg_all.get_qubit(qubit)
cfg.update([
    ('steps', 101),
    ('qb_gain_ge', QickSweep1D('gainloop', 0.0, 1.0)),
])
rabi_result = PowerRabi(cfg).run(py_avg=5)
show(rabi_result, title='Power Rabi', xlabel='Gain (a.u.)')

pi_gain = rabi_result.get_param('pi_gain')
pi2_gain = rabi_result.get_param('pi2_gain')
if pi_gain is not None:
    cfg_all.update('pi_gain_ge', pi_gain, q_index=qubit)
if pi2_gain is not None:
    cfg_all.update('pi2_gain_ge', pi2_gain, q_index=qubit)


## Step 4 — T1 Relaxation (`T1`)

Prepares |e⟩ with a π pulse, then waits a variable delay before readout.
Exponential decay fit yields the energy relaxation time T1.

In [ ]:
from QickworkspaceV2.experiments.coherence import T1

cfg = cfg_all.get_qubit(qubit)
cfg.update([
    ('steps', 101),
    ('wait_time', QickSweep1D('waitloop', 0.0, 200.0)),
])
t1_result = T1(cfg).run(py_avg=5)
show(t1_result, title='T1 Relaxation', xlabel='Wait time (us)')

T1_us = t1_result.get_param('T1_us')
print(f'T1 = {T1_us:.1f} us' if T1_us is not None else 'T1 fit failed')


## Step 5 — Ramsey (`Ramsey`)

Two π/2 pulses bracketing a variable free-evolution time.  The oscillation
frequency reveals any detuning from the true qubit frequency; the envelope
decay gives T2\*.

In [ ]:
from QickworkspaceV2.experiments.coherence import Ramsey

cfg = cfg_all.get_qubit(qubit)
cfg.update([
    ('steps', 101),
    ('wait_time', QickSweep1D('waitloop', 0.0, 5.0)),
    ('virtual_detune', 1.0),
])
ram_result = Ramsey(cfg).run(py_avg=5)
show(ram_result, title='Ramsey', xlabel='Free evolution time (us)')

T2r = ram_result.get_param('T2r_us')
print(f'T2* = {T2r:.1f} us' if T2r is not None else 'Ramsey fit failed')


## Summary

In [ ]:
from QickworkspaceV2 import QualityFlag

results = [
    ('ResonatorSpec', res_result),
    ('QubitSpec',     qs_result),
    ('PowerRabi',     rabi_result),
    ('T1',            t1_result),
    ('Ramsey',        ram_result),
]

print(f'{"Experiment":<20} {"Quality":<12} {"Scalar result"}')
print('-' * 50)
for name, r in results:
    val = f'{r.scalar_result:.4f}' if r.scalar_result is not None else 'N/A'
    print(f'{name:<20} {str(r.quality.value):<12} {val}')

**Next:** [03_batch_pipeline.ipynb](03_batch_pipeline.ipynb) — composing experiments into an ordered pipeline.